In [ ]:
import os
import shutil
import glob

# 1. Copy your modified codebase to the working directory so you can run it
shutil.copytree('/kaggle/input/datasets/youssefmansourrr/diffam-repo/DIFFAM', '/kaggle/working/DiffAM', dirs_exist_ok=True)
os.chdir('/kaggle/working/DiffAM')

# 2. Install requirements
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q gdown lpips

# 3. Create missing directories
os.makedirs('/kaggle/working/DiffAM/checkpoint', exist_ok=True)
os.makedirs('/kaggle/working/DiffAM/assets/datasets', exist_ok=True)

# 4. Download Target Models & MT-Dataset
if not os.path.exists('/kaggle/working/DiffAM/assets/models'):
    print("Downloading target FR models...")
    !gdown "1IKiWLv99eUbv3llpj-dOegF3O7FWW29J" -O /kaggle/working/assets.zip
    !unzip -o -q /kaggle/working/assets.zip -d /kaggle/working/assets_extracted/
    shutil.move('/kaggle/working/assets_extracted/assets', '/kaggle/working/DiffAM/assets_temp')
    shutil.copytree('/kaggle/working/DiffAM/assets_temp', '/kaggle/working/DiffAM/assets', dirs_exist_ok=True)
    !rm -rf /kaggle/working/assets.zip /kaggle/working/assets_extracted /kaggle/working/DiffAM/assets_temp

# 5. Symlink Kaggle's CelebAMask-HQ dataset instantly
celeb_path = glob.glob('/kaggle/input/datasets/ipythonx/celebamaskhq/CelebAMask-HQ', recursive=True)[0]
target_link = '/kaggle/working/DiffAM/assets/datasets/CelebAMask-HQ'
if not os.path.exists(target_link):
    os.symlink(celeb_path, target_link)

print("Environment setup and data linking complete.")

In [ ]:
import os

os.chdir('/kaggle/working/DiffAM')
os.makedirs('pretrained', exist_ok=True)

print("Downloading base pretrained weights from official repository...")
# Download the author's Google Drive folder directly
!gdown --folder "1L8caY-FVzp9razKMuAt37jCcgYh3fjVU" -O /kaggle/working/DiffAM/pretrained_dl

# Move files out of the downloaded subfolder into the target directory
!mv /kaggle/working/DiffAM/pretrained_dl/*/* /kaggle/working/DiffAM/pretrained/ 2>/dev/null || mv /kaggle/working/DiffAM/pretrained_dl/* /kaggle/working/DiffAM/pretrained/

# Clean up
!rm -rf /kaggle/working/DiffAM/pretrained_dl

print("Weights downloaded successfully! Contents:")
print(os.listdir('pretrained'))

In [ ]:
# 1. Forcefully rip out Kaggle's pre-installed Hugging Face library
!pip uninstall -y datasets -q

In [ ]:
import os, torch
os.chdir('/kaggle/working/DiffAM')
torch.cuda.empty_cache()

!PYTHONPATH=/kaggle/working/DiffAM python main.py \
  --makeup_transfer \
  --config celeba.yml \
  --exp /kaggle/working/DiffAM/runs/test \
  --do_train 1 --do_test 1 \
  --n_train_img 200 --n_test_img 100 \
  --n_iter 6 --t_0 60 \
  --n_inv_step 20 --n_train_step 6 --n_test_step 6 \
  --lr_clip_finetune 4e-6 \
  --MT_iter_without_adv 2 \
  --MT_adv_loss_w 1.2 \
  --model_path pretrained/celeba_hq.ckpt \
  --target_img 3 \
  --target_model 3 \
  --ref_img 'XMY-060'

# Save files


In [ ]:
import os
import glob

# Make sure we are in the right directory
os.chdir('/kaggle/working/DiffAM')

# 1. Clean out the useless intermediate checkpoints to keep the download small
print("Cleaning up old checkpoints to save space...")
checkpoints = glob.glob('checkpoint/*.pth')
checkpoints.sort() # Sorts them so the final iteration is at the end of the list

# Keep only the final checkpoint, delete the rest
if len(checkpoints) > 1:
    for ckpt in checkpoints[:-1]:
        os.remove(ckpt)
        print(f"🗑️ Deleted intermediate weight: {ckpt}")

# 2. Define ALL the critical folders you need for your thesis
folders_to_save = "precomputed checkpoint sample_fake sample_real sample_fake_test sample_real_test runs"

print("\nZipping all necessary files... This will take a minute...")

# 3. Zip everything into the main Kaggle working directory
!zip -r -q /kaggle/working/diffam_final_results.zip {folders_to_save}

print("✅ Zipping complete! You can now download 'diffam_final_results.zip' from the Kaggle sidebar.")